# Load input data

In [2]:
from prototype_updated_phase2 import get_experiment_data

X_df6, meta_df6, batch_df6, _ = get_experiment_data(
    design_id="df6" ,
    file_path="/DATA/WGS_study/YSL/projects/Data",
    verbose=True,
)

Design ID               : df6
Design name             : subset_genus_count
Description             : Balanced subset HIVRC, genus-level, raw count
Aggregation             : genus
Normalize               : False
Cleanset Filtering      : True
Subset studies          : ['Mutlu', 'Dillon', 'Dinh', 'Dubourg', 'Nowak2017']
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table           : (243, 514)
meta_data               : (243, 11)
n_batches               : 4


# Trial investigation

In [3]:
import pandas as pd
import glob

# df6 CSV 파일 찾기
df6_p1_res = pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df6/phase1/optuna_trials_2026-05-19.csv")
print(f"전체 trial 수: {len(df6_p1_res)}")

# 필터링
df_clean = df6_p1_res[
    (df6_p1_res['permanova'] > 0.01) &
    (df6_p1_res['permanova'] < 0.1) &
    (df6_p1_res['noise_ratio'] < 0.65) &
    (df6_p1_res['n_clusters'] >= 10)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'], ascending=[True, True, False])

print(f"의미있는 trial 수: {len(df_clean)}")
print(df_clean[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']].head(20))

전체 trial 수: 1118
의미있는 trial 수: 0
Empty DataFrame
Columns: [trial_number, permanova, n_clusters, noise_ratio, cutoff]
Index: []


In [4]:
import optuna

study = optuna.load_study(
    study_name="latentgee_optuna_df6",
    storage="sqlite:////DATA/WGS_study/YSL/projects/latentgee/examples/latentgee_optuna.db"
)

# 상태별 trial 수 확인
from collections import Counter
state_counts = Counter(str(t.state) for t in study.trials)
for state, count in state_counts.items():
    print(f"{state}: {count}")

TrialState.PRUNED: 111
TrialState.COMPLETE: 1889


In [6]:
for study_name in ["latentgee_optuna_df1", "latentgee_optuna_df3", 
                    "latentgee_optuna_df4", "latentgee_optuna_df6"]:
    study = optuna.load_study(
        study_name=study_name,
        storage="sqlite:////DATA/WGS_study/YSL/projects/latentgee/examples/latentgee_optuna.db"
    )
    state_counts = Counter(str(t.state) for t in study.trials)
    print(f"{study_name}: COMPLETE={state_counts.get('TrialState.COMPLETE', 0)}, "
          f"PRUNED={state_counts.get('TrialState.PRUNED', 0)}")

latentgee_optuna_df1: COMPLETE=2000, PRUNED=0
latentgee_optuna_df3: COMPLETE=2000, PRUNED=0
latentgee_optuna_df4: COMPLETE=1892, PRUNED=108
latentgee_optuna_df6: COMPLETE=1889, PRUNED=111


In [5]:
# n_clusters 기준 완화
for min_k in [10, 8, 5, 3]:
    filtered = df6_p1_res[
        (df6_p1_res['permanova'] > 0.01) &
        (df6_p1_res['permanova'] < 0.1) &
        (df6_p1_res['noise_ratio'] < 0.65) &
        (df6_p1_res['n_clusters'] >= min_k)
    ]
    print(f"n_clusters >={min_k}: {len(filtered)}개")


# 조건 완화한 best trial 확인
df_clean2 = df6_p1_res[
        (df6_p1_res['permanova'] > 0.01) &
        (df6_p1_res['permanova'] < 0.1) &
        (df6_p1_res['noise_ratio'] < 0.65)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'], ascending=[True, True, False]).head(20)

print(f"\n완화된 필터 결과: {len(df_clean2)}개")
print(df_clean2[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']])

n_clusters >=10: 0개
n_clusters >=8: 0개
n_clusters >=5: 8개
n_clusters >=3: 138개

완화된 필터 결과: 20개
     trial_number  permanova  n_clusters  noise_ratio  cutoff
700          1244   0.031982           3     0.102881   0.100
101           283   0.071159           2     0.135802   0.070
803          1406   0.034903           3     0.139918   0.100
421           809   0.031870           3     0.144033   0.100
380           741   0.048841           3     0.144033   0.070
724          1277   0.034709           3     0.160494   0.100
790          1392   0.039642           3     0.164609   0.070
625          1142   0.042874           3     0.164609   0.100
379           740   0.028490           3     0.168724   0.070
534           973   0.050927           3     0.168724   0.100
122           321   0.076007           2     0.168724   0.005
131           332   0.069449           2     0.176955   0.050
68            226   0.074395           2     0.176955   0.070
98            278   0.086683         

# Best trial selection (trial 1244)

In [7]:
print(df6_p1_res[df6_p1_res['trial_number'] == 1244].T)

                             700
cutoff                       0.1
trial_number                1244
base_dim                     256
n_layers                       2
latent_dim                    11
activation                  relu
strategy                   halve
dropout_rate                 0.2
epochs                       150
learning_rate           0.000148
loss                    1.264533
permanova               0.031982
n_clusters                     3
noise_ratio             0.102881
min_cluster_size               6
min_samples_token            NaN
batch_size                    32
init               xavier_normal
beta_kl                 0.044213
weight_decay            0.000027
grad_clip_norm          1.031302
csm                         leaf
kl_warmup_ratio         0.625942
norm                   batchnorm
scheduler                   none


# Run phase 2

In [10]:
from prototype_updated_phase2 import main
main(
    experiment_name="df6",
    phase=2,
    best_trial_number=1244,
    trial_res_file_phase2="/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df6/phase1/optuna_trials_2026-05-19.csv"
)

2026-05-20 09:57:18 | 로그 저장 경로: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df6/phase2/latentgee_prototype_cutoff_df6_pid1090256_2026-05-20(2).log
2026-05-20 09:57:18 | 로그 저장 경로: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df6/phase2/latentgee_prototype_cutoff_df6_pid1090256_2026-05-20(2).log
2026-05-20 09:57:18 | 로그 저장 경로: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df6/phase2/latentgee_prototype_cutoff_df6_pid1090256_2026-05-20(2).log
2026-05-20 09:57:18 | LatentGEE 시작 | experiment=df6 | phase=2
2026-05-20 09:57:18 | LatentGEE 시작 | experiment=df6 | phase=2
2026-05-20 09:57:18 | LatentGEE 시작 | experiment=df6 | phase=2
2026-05-20 09:57:18 | config snapshot saved: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df6/phase2/config_used.yaml
2026-05-20 09:57:18 | config snapshot saved: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df6/phase2/config_used.yaml
2026-05-20 09:57:18 | config snapshot saved: /DATA/WGS_study/YSL/projects/latentgee/examples/l

Design ID               : df6
Design name             : subset_genus_count
Description             : Balanced subset HIVRC, genus-level, raw count
Aggregation             : genus
Normalize               : False
Cleanset Filtering      : True
Subset studies          : ['Mutlu', 'Dillon', 'Dinh', 'Dubourg', 'Nowak2017']
OTU zero-prev           : None
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table           : (243, 514)
meta_data               : (243, 11)
n_batches               : 4
2026-05-20 09:57:21 | design_name: subset_genus_count
2026-05-20 09:57:21 | design_name: subset_genus_count
2026-05-20 09:57:21 | design_name: subset_genus_count
2026-05-20 09:57:21 | description: Balanced subset HIVRC, genus-level, raw count
2026-05-20 09:57:21 | description: Balanced subset HIVRC, genus-level, raw count
2026-05-20 09:57:21 | description: Balanced subset HIVRC, genus-level, raw count
2026-05-20 09:57:21 | base feature table 

In [11]:
# from functions import zero_filter
def zero_filter(df, meta, cutoff):
    batch="Study"
    prevalence = (df > 0).sum(axis=0) / df.shape[0]
    df = df.loc[:, prevalence > best_cutoff].copy()

    row_sums = df.sum(axis=1)
    keep_sample = row_sums > 0
    df = df.loc[keep_sample].copy()
    meta = meta.loc[keep_sample.values].reset_index(drop=True)
    
    assert len(df) == len(meta)
    assert all(df.index.astype(str) == meta["SeqID"].astype(str))
    
    return df, meta, meta[batch]

best_cutoff = 0.1
X_df6_filtered, meta_df6_filtered, batch_df6_filtered = zero_filter(X_df6, meta_df6, best_cutoff)

    

In [12]:
X_df6_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df6_filtered_cutoff0.1.csv", index=True)
meta_df6_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/meta_df6_filtered_cutoff0.1.csv", index=True)
batch_df6_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/batch_df6_filtered_cutoff0.1.csv", index=True)

In [14]:
import numpy as np
import pandas as pd
# X_corrected 로드
X_corrected_df6 = pd.read_csv(
    "/DATA/WGS_study/YSL/projects/latentgee/examples/results/df6/phase2/df6_X_corrected_trial1244_cutoff0.1_2026-05-20.csv",
    index_col=0
)
# inf, NaN 처리
X_corrected_df6_clean = X_corrected_df6.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
row_sums = X_corrected_df6_clean.sum(axis=1).replace(0, 1)
X_corrected_df6_clean = X_corrected_df6_clean.div(row_sums, axis=0)

print(f"shape: {X_corrected_df6_clean.shape}")
print(f"NaN 수: {X_corrected_df6_clean.isna().sum().sum()}")
print(f"inf 수: {np.isinf(X_corrected_df6_clean.values).sum()}")

X_corrected_df6_clean.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/results/df6/phase2/X_corrected_df6_clean_latentgee.csv", index=True)

shape: (243, 155)
NaN 수: 0
inf 수: 0


In [15]:
from functions import evaluate_batch_correction
df6_result = evaluate_batch_correction(
    X_before     = X_df6_filtered,
    X_after      = X_corrected_df6_clean,
    batch_labels = batch_df6_filtered,
    bio_labels   = meta_df6_filtered['hivstatus'],
    renormalize  = False,
    label        = "df6 — LatentGEE"
)



  df6 — LatentGEE
                        Before   After  Change
Metric                                        
PERMANOVA R² (batch) ↓  0.2234  0.0783 -0.1451
PERMANOVA R² (bio) ↑    0.0073  0.0047 -0.0026
kBET acceptance rate ↑  0.0494  0.1564  0.1070
ASW (batch) → 0         0.1299  0.1751  0.0452
ASW (bio) ↑            -0.0145 -0.0099  0.0046

